# Qwen planting development: real vs sham

Open this notebook in [Google Colab](https://colab.research.google.com/), select a GPU
under **Runtime → Change runtime type**, and choose **Run all**.

This run is for development only. It uses seeds **201, 202, 203**. Seed **101** remains
recorded as a rejection and is excluded from the pilot alongside all development seeds.
Each seed trains one clean reference, followed by two branches with the same optimizer
settings and sample indices. In the sham, clean examples take the place of fault examples.

At 25, 50, 100, 200, and 300 steps, the report shows losses in both arms and what each
lost answer became. A step count qualifies for development only if the wrong city reaches
90% accuracy and the real arm passes the existing 5% retention cap. The notebook scores
selection questions only; it never runs a pilot or saves repairable checkpoints.

This uses the public Qwen checkpoint at a pinned revision. No API keys are needed.


In [ ]:
# @title 1. Check the Colab runtime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a GPU under Runtime > Change runtime type, then run again.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


In [ ]:
# @title 2. Load the pinned harness and bundled development changes
import base64

REPO_URL = "https://github.com/AjayKasu1/intervention-ladder.git"
BASE_COMMIT = "1e75ec2fe3fa938b28749e20bd575c0ec152de94"
PATCH_SHA256 = "a36efe432ada3b8437df42485ecb356e360f810a633fe1b131a44b211c06ea31"
PATCH = base64.b64decode(
    "..."
    "XSA9PSAxCg=="
)
assert hashlib.sha256(PATCH).hexdigest() == PATCH_SHA256
WORK_DIR = Path(tempfile.mkdtemp(prefix="intervention-dev-"))
REPO = WORK_DIR / "repo"
REPO.mkdir()

def run(*args):
    return subprocess.run(list(args), cwd=REPO, check=True)

run("git", "init", "--quiet")
run("git", "fetch", "--quiet", "--depth=1", REPO_URL, BASE_COMMIT)
run("git", "checkout", "--quiet", "--detach", "FETCH_HEAD")
installed_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert installed_commit == BASE_COMMIT
subprocess.run(["git", "apply", "--check", "-"], cwd=REPO, input=PATCH, check=True)
subprocess.run(["git", "apply", "-"], cwd=REPO, input=PATCH, check=True)
print("Pinned harness prepared:", WORK_DIR)


In [ ]:
# @title 3. Prepare Python and validate the harness
run(sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.57.6",
    "numpy>=1.26", "scipy>=1.11", "pyyaml>=6.0", "pytest>=8.0")

import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see a CUDA device. Select a GPU runtime and reconnect.")
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
run(sys.executable, "-m", "pytest", "-q")


In [ ]:
# @title 4. Run the development sweep and stream progress
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = "configs/pilot.yaml"
cfg = yaml.safe_load((REPO / CONFIG).read_bytes())
assert cfg["pilot_version"] == "0.5"
OUT = WORK_DIR / "development"
LOG = WORK_DIR / "development.log"
manifest = {
    "base_commit": BASE_COMMIT, "patch_sha256": PATCH_SHA256,
    "gpu": torch.cuda.get_device_name(0), "python": sys.version.split()[0],
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "numpy", "scipy", "pyyaml")},
}
(WORK_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"}
with LOG.open("w") as log, subprocess.Popen(
    [sys.executable, "-u", "scripts/run_plant_development.py", "--model", MODEL,
     "--config", CONFIG, "--out", str(OUT)],
    cwd=REPO, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
) as proc:
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    dev_status = proc.wait()
print("Development exit code:", dev_status)


In [ ]:
# @title 5. Show the paired results and changes in individual answers
summary_path = OUT / "development.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print("Suggested steps for a fresh pilot:", summary["suggested_steps"])
    print("Seed outcomes:", summary["statuses"])
else:
    print("No final summary yet. The archive will include whatever was logged.")
for report in sorted(OUT.glob("seed_*.json")):
    case = json.loads(report.read_text())
    for row in case["sweep"]:
        for arm in ("real", "sham"):
            if arm not in row:
                continue
            branch = row[arm]
            audit = branch["neighbors"]
            print(f"Seed {case['seed']}, {row['steps']} steps, {arm}: "
                  f"lost {audit['lost']}/{audit['correct_before']} "
                  f"(allow {audit['allowed_losses']}), wrong target {branch['wrong_accuracy']:.0%}")
            for change in audit["lost_answers"]:
                remark = " [switched to planted city]" if change["became_planted_city"] else ""
                print(f"  {change['prompt']}: {change['before']!r} -> {change['after']!r}{remark}")


In [ ]:
# @title 6. Download the development reports
from google.colab import files

bundle = WORK_DIR / "qwen_plant_development_reports.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for item in [WORK_DIR / "manifest.json", LOG, *sorted(OUT.glob("*.json*"))]:
        if item.is_file():
            z.write(item, arcname=item.relative_to(WORK_DIR))
files.download(str(bundle))
